In [ ]:
# Install required packages
# !pip install langchain langchain-openai langgraph chromadb

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain.tools import Tool
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationBufferMemory

load_dotenv()
print("✅ Setup complete")

<a id="part1"></a>
## Part 1: Framework Overview

### Popular Agent Frameworks

| Framework | Best For | Learning Curve |
|-----------|----------|----------------|
| **LangChain** | General-purpose agents, quick prototypes | Easy |
| **LangGraph** | Complex workflows, cyclic graphs | Medium |
| **CrewAI** | Multi-agent teams, role-based agents | Easy |
| **AutoGPT** | Autonomous task completion | Medium |
| **Custom** | Full control, specific requirements | Hard |

### When to Use Each

**LangChain:**
- ✅ Quick prototypes
- ✅ Standard agent patterns
- ✅ Rich ecosystem of tools
- ❌ Limited control over agent loop

**LangGraph:**
- ✅ Complex state machines
- ✅ Cyclic workflows
- ✅ Human-in-the-loop
- ❌ More boilerplate code

**Custom Implementation:**
- ✅ Full control
- ✅ Optimized for specific use case
- ❌ More development time

<a id="part2"></a>
## Part 2: LangChain Agents

### Building Your First LangChain Agent

In [ ]:
# Initialize LLM
llm = ChatOpenAI(model="gpt-4", temperature=0)

# Define tools
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    return len(word)

def multiply_numbers(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

# Create LangChain tools
tools = [
    Tool(
        name="get_word_length",
        func=get_word_length,
        description="Get the length of any word. Input should be a single word."
    ),
    Tool(
        name="multiply",
        func=lambda x: multiply_numbers(*map(float, x.split(','))),
        description="Multiply two numbers. Input should be two numbers separated by comma, e.g., '5,3'"
    )
]

print(f"✅ Created {len(tools)} tools")

In [ ]:
# Create agent prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with access to tools."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# Create agent
agent = create_openai_functions_agent(llm, tools, prompt)

# Create agent executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=5
)

print("✅ LangChain agent ready")

In [ ]:
# Test the agent
response = agent_executor.invoke({
    "input": "What is the length of the word 'LangChain' multiplied by 3?"
})

print(f"\n🤖 Agent Response: {response['output']}")

### Real-World Example: Research Agent

In [ ]:
import wikipedia
from datetime import datetime

# Wikipedia search tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia and return a summary."""
    try:
        return wikipedia.summary(query, sentences=3)
    except:
        return f"Could not find information about '{query}'"

# Current date tool
def get_current_date() -> str:
    """Get the current date."""
    return datetime.now().strftime("%B %d, %Y")

# Calculator tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        result = eval(expression)
        return str(result)
    except:
        return "Invalid expression"

# Create research tools
research_tools = [
    Tool(
        name="wikipedia",
        func=search_wikipedia,
        description="Search Wikipedia for information. Input should be a search query."
    ),
    Tool(
        name="current_date",
        func=get_current_date,
        description="Get today's date. No input required."
    ),
    Tool(
        name="calculator",
        func=calculate,
        description="Calculate mathematical expressions. Input should be a valid math expression."
    )
]

# Create research agent
research_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research assistant. Answer questions using available tools."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

research_agent = create_openai_functions_agent(llm, research_tools, research_prompt)
research_executor = AgentExecutor(
    agent=research_agent,
    tools=research_tools,
    verbose=True
)

print("✅ Research agent ready")

In [ ]:
# Test research agent
result = research_executor.invoke({
    "input": "Who invented Python programming language and when?"
})

print(f"\n📚 Research Result:\n{result['output']}")

<a id="part3"></a>
## Part 3: LangGraph Workflows

LangGraph enables more complex, graph-based agent architectures.

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

# Define state
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    next_step: str
    final_answer: str

# Node functions
def planning_node(state: AgentState) -> AgentState:
    """Plan the approach"""
    print("🧠 Planning...")
    state["messages"].append("Created plan")
    state["next_step"] = "research"
    return state

def research_node(state: AgentState) -> AgentState:
    """Conduct research"""
    print("🔍 Researching...")
    state["messages"].append("Gathered information")
    state["next_step"] = "synthesis"
    return state

def synthesis_node(state: AgentState) -> AgentState:
    """Synthesize findings"""
    print("✍️ Synthesizing...")
    state["final_answer"] = "Research complete with findings"
    state["next_step"] = "end"
    return state

# Build graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("planning", planning_node)
workflow.add_node("research", research_node)
workflow.add_node("synthesis", synthesis_node)

# Add edges
workflow.set_entry_point("planning")
workflow.add_edge("planning", "research")
workflow.add_edge("research", "synthesis")
workflow.add_edge("synthesis", END)

# Compile
app = workflow.compile()

print("✅ LangGraph workflow created")

In [ ]:
# Run the workflow
initial_state = {
    "messages": ["Starting research task"],
    "next_step": "planning",
    "final_answer": ""
}

final_state = app.invoke(initial_state)

print("\n📊 Workflow Result:")
print(f"Messages: {final_state['messages']}")
print(f"Final Answer: {final_state['final_answer']}")

<a id="part4"></a>
## Part 4: Memory Integration

Add memory to agents for context-aware conversations.

In [ ]:
from langchain.memory import ConversationBufferMemory

# Create memory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Create agent with memory
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Remember previous conversation."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

memory_agent = create_openai_functions_agent(llm, research_tools, memory_prompt)
memory_executor = AgentExecutor(
    agent=memory_agent,
    tools=research_tools,
    memory=memory,
    verbose=True
)

print("✅ Agent with memory ready")

In [ ]:
# Test memory
print("First message:")
response1 = memory_executor.invoke({"input": "My name is Alice"})
print(response1['output'])

print("\nSecond message (should remember name):")
response2 = memory_executor.invoke({"input": "What's my name?"})
print(response2['output'])

<a id="part5"></a>
## Part 5: Framework Comparison

### LangChain vs Custom Implementation

In [ ]:
import time

# LangChain approach (5-10 lines)
def langchain_agent():
    tools = [Tool(name="calc", func=lambda x: eval(x), description="Calculator")]
    agent = create_openai_functions_agent(llm, tools, prompt)
    executor = AgentExecutor(agent=agent, tools=tools)
    return executor.invoke({"input": "What is 15 + 27?"})

# Custom approach (50+ lines)
def custom_agent():
    # Would need: prompt engineering, tool execution, loop control,
    # error handling, parsing, etc.
    pass

print("✅ LangChain: Quick to build, less control")
print("✅ Custom: More code, full control")

### Decision Matrix

| Criteria | LangChain | LangGraph | Custom |
|----------|-----------|-----------|--------|
| **Development Speed** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐ |
| **Flexibility** | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Documentation** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | N/A |
| **Learning Curve** | ⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Community Support** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐ |
| **Production Ready** | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |

<a id="part6"></a>
## Part 6: Production Patterns

### Error Handling & Retries

In [ ]:
from langchain.callbacks import StdOutCallbackHandler

# Add custom error handling
class SafeAgentExecutor(AgentExecutor):
    def _call(self, inputs, **kwargs):
        try:
            return super()._call(inputs, **kwargs)
        except Exception as e:
            return {
                "output": f"Error occurred: {str(e)}",
                "error": True
            }

print("✅ Safe agent executor ready")

### Monitoring & Logging

In [ ]:
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class LoggingAgentExecutor(AgentExecutor):
    def _call(self, inputs, **kwargs):
        logger.info(f"Agent input: {inputs}")
        result = super()._call(inputs, **kwargs)
        logger.info(f"Agent output: {result}")
        return result

print("✅ Logging configured")

## 🎯 Knowledge Check

**Q1:** When should you use LangChain vs custom implementation?  
**Q2:** What's the main advantage of LangGraph?  
**Q3:** How does memory work in LangChain agents?

<details>
<summary>Click for answers</summary>

**A1:** LangChain for rapid prototyping and standard patterns; custom for full control and optimization  
**A2:** Graph-based workflows with cycles, branching, and complex state management  
**A3:** Memory stores conversation history and passes it to the agent as context  
</details>

## 🚀 Next Steps

1. Complete the **Agent Framework Challenge**
2. Read Notebook 5: **Multi-Agent Systems**
3. Build a production agent with LangChain
4. Experiment with LangGraph for complex workflows

---

**Great work! You now know how to leverage frameworks to build agents faster! 🎉**